# ЛР3. Основной XML-RPC сервер

Тетрадь подготовлена на основе предыдущей ЛР и требований ЛР3.

In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import xmlrpc.client
import datetime
import pickle
import numpy as np
import pandas as pd

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", 8008), requestHandler=RequestHandler, allow_none=True)
server.register_introspection_functions()

def ping():
    return True
server.register_function(ping, 'ping')

def now():
    return datetime.datetime.now()
server.register_function(now, 'now')

def show_type(arg):
    return (str(arg), str(type(arg)), arg)
server.register_function(show_type, 'show_type')

def test_sum(a, b):
    return a + b
server.register_function(test_sum, 'test_sum')

def test_pow(a, b):
    return a ** b
server.register_function(test_pow, 'test_pow')

def _levenshtein(a, b):
    if len(a) < len(b):
        a, b = b, a
    if len(b) == 0:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        cur = [i + 1]
        for j, cb in enumerate(b):
            cur.append(min(prev[j + 1] + 1, cur[j] + 1, prev[j] + (ca != cb)))
        prev = cur
    return prev[-1]

def _close(a, b):
    return _levenshtein(a.strip().lower(), b.strip().lower()) <= 1

def black_list_check(surname, name, patr, birth_date):
    try:
        frame = pd.read_csv('bad_boys2.csv', header=0, sep=',', encoding='utf-8')
    except FileNotFoundError:
        return "Error: 'bad_boys2.csv' not found."
    surname, name, patr, birth_date = map(str.strip, (surname, name, patr, birth_date))
    exact_matches, similar_matches = [], []
    for _, row in frame.iterrows():
        s, n, p, b = map(str.strip, (str(row['Surname']), str(row['Name']), str(row['Patronym']), str(row['Birth'])))
        if s.lower() == surname.lower() and n.lower() == name.lower() and p.lower() == patr.lower() and b == birth_date:
            exact_matches.append(f"{s} {n} {p} {b}")
        elif _close(s, surname) and _close(n, name) and _close(p, patr) and b == birth_date:
            similar_matches.append(f"{s} {n} {p} {b}")
    if exact_matches:
        return f"{surname} {name} {patr} {birth_date} : yes"
    if similar_matches:
        return similar_matches + ["similar"]
    return f"{surname} {name} {patr} {birth_date} : no"
server.register_function(black_list_check, 'black_list_check')

def _to_gray(img_arr):
    if img_arr.ndim == 3:
        return (0.299 * img_arr[:, :, 0] + 0.587 * img_arr[:, :, 1] + 0.114 * img_arr[:, :, 2]).astype(np.uint8)
    return img_arr

def send_back_inversion(bin_data):
    img_arr = pickle.loads(bin_data.data)
    return xmlrpc.client.Binary(pickle.dumps(255 - img_arr))
server.register_function(send_back_inversion, 'color_inversion')

def binarize_image(bin_data, threshold):
    img_arr = pickle.loads(bin_data.data)
    gray = _to_gray(img_arr)
    bw = np.where(gray > threshold, 255, 0).astype(np.uint8)
    if img_arr.ndim == 3:
        bw = np.stack([bw, bw, bw], axis=2)
    return xmlrpc.client.Binary(pickle.dumps(bw))
server.register_function(binarize_image, 'binarize_image')

def flip_image_vertical(bin_data):
    img_arr = pickle.loads(bin_data.data)
    flipped = img_arr[:, ::-1, ...]
    return xmlrpc.client.Binary(pickle.dumps(flipped))
server.register_function(flip_image_vertical, 'flip_vertical')

def cloudiness_percent(bin_data, threshold):
    img_arr = pickle.loads(bin_data.data)
    gray = _to_gray(img_arr)
    return float(np.sum(gray > threshold) / gray.size * 100.0)
server.register_function(cloudiness_percent, 'cloudiness_percent')

print("Основной сервер: порт 8008")
server.serve_forever()


Основной сервер: порт 8008


127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:48] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:53] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:53] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:53] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:53] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202